# Flow Matching from Scratch

A hands-on tutorial: theory, training, and animated sampling — learning to generate **collision-free robot arm configurations** via flow matching.

---

## 1. The core idea

We want to learn a transport map that pushes a simple **source distribution** $p_0$ (standard Gaussian) to a complex **target distribution** $p_1$ (our data).

Flow matching constructs a time-dependent **velocity field** $v_\theta(x, t)$ that defines an ODE:

$$\frac{dx}{dt} = v_\theta(x_t,\; t), \qquad t \in [0, 1]$$

Integrating from $t{=}0$ to $t{=}1$ transports noise samples $x_0 \sim \mathcal{N}(0, I)$ into data samples $x_1 \sim p_\text{data}$.

---

## 2. Conditional flow matching (CFM)

Directly regressing onto the marginal velocity field is intractable. Instead we use **conditional** paths — one per data point $x_1$:

$$x_t = (1 - t)\, x_0 + t\, x_1, \qquad x_0 \sim \mathcal{N}(0, I)$$

The conditional velocity along this path is simply:

$$u_t(x_t \mid x_1) = x_1 - x_0$$

We train a neural network $v_\theta$ to predict this velocity:

$$\mathcal{L}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1],\; x_1 \sim p_\text{data},\; x_0 \sim \mathcal{N}(0,I)} \Big[\lVert v_\theta(x_t, t) - (x_1 - x_0) \rVert^2 \Big]$$

At inference we solve the ODE with Euler steps:

$$x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

## 3. Setup & data

In [ ]:
!pip install -q scienceplots

In [ ]:
import tqdm
import math
import torch
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import scienceplots

plt.style.use(["science", "ieee", "no-latex"])
plt.rcParams["figure.dpi"] = 150        # notebook display
plt.rcParams["savefig.dpi"] = 300       # saved files at publication quality

# Only domain-specific visuals that the style doesn't cover
CSPACE_CMAP = ListedColormap(["white", "#e8e0dc"])
C_OBS  = "#e8d8d2"
C_OBS_E = "#b09080"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### Target distribution: collision-free C-space of a 2-link arm

A planar robot arm with links $L_1 = 1.5$, $L_2 = 1.0$ operates in a workspace with circular obstacles.
A configuration $(\theta_1, \theta_2)$ is **valid** if neither link segment intersects any obstacle.

We sample $N$ valid configurations uniformly — the resulting distribution in **configuration space** (C-space) has beautiful, non-convex shapes carved by the C-space obstacles. Learning to sample from this free space is directly useful for **sampling-based motion planning**.

In [ ]:
# --- 2-link planar robot arm ---
L1, L2 = 1.5, 1.0

# Workspace obstacles: (center, radius)
obstacles_ws = [
    (np.array([1.5, 1.0]),  0.50),
    (np.array([-0.8, 1.6]), 0.45),
    (np.array([0.6, -1.4]), 0.40),
]


def forward_kin(t1, t2):
    """Elbow and end-effector positions for joint angles (t1, t2)."""
    elbow = np.array([L1 * np.cos(t1), L1 * np.sin(t1)])
    ee = elbow + np.array([L2 * np.cos(t1 + t2), L2 * np.sin(t1 + t2)])
    return elbow, ee


def seg_circle_dist(ax, ay, bx, by, cx, cy):
    """Min distance from point (cx,cy) to segment (ax,ay)-(bx,by). Vectorized."""
    dx, dy = bx - ax, by - ay
    len_sq = dx**2 + dy**2
    t = np.clip(((cx - ax) * dx + (cy - ay) * dy) / (len_sq + 1e-12), 0, 1)
    px, py = ax + t * dx, ay + t * dy
    return np.sqrt((cx - px)**2 + (cy - py)**2)


def is_free(t1, t2):
    """Scalar collision check for rejection sampling."""
    elbow, ee = forward_kin(t1, t2)
    for c, r in obstacles_ws:
        d1 = seg_circle_dist(0, 0, elbow[0], elbow[1], c[0], c[1])
        d2 = seg_circle_dist(elbow[0], elbow[1], ee[0], ee[1], c[0], c[1])
        if d1 < r or d2 < r:
            return False
    return True


# --- Precompute C-space collision map (vectorized) ---
_res = 300
_t1g = np.linspace(-np.pi, np.pi, _res)
_t2g = np.linspace(-np.pi, np.pi, _res)
T1G, T2G = np.meshgrid(_t1g, _t2g)
_ex = L1 * np.cos(T1G);  _ey = L1 * np.sin(T1G)
_eex = _ex + L2 * np.cos(T1G + T2G);  _eey = _ey + L2 * np.sin(T1G + T2G)

free_grid = np.ones_like(T1G, dtype=bool)
for c, r in obstacles_ws:
    free_grid &= (seg_circle_dist(0, 0, _ex, _ey, c[0], c[1]) >= r)
    free_grid &= (seg_circle_dist(_ex, _ey, _eex, _eey, c[0], c[1]) >= r)

cspace_bg = (~free_grid).astype(float)

PI = np.pi
EXTENT = (-PI, PI, -PI, PI)
PLIM = (-PI - 0.3, PI + 0.3)


def draw_cspace_bg(ax, alpha=0.7):
    ax.imshow(cspace_bg, extent=EXTENT, origin="lower",
              cmap=CSPACE_CMAP, alpha=alpha, aspect="auto")


def style_cspace_ax(ax):
    ax.set_xlim(*PLIM); ax.set_ylim(*PLIM)
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
    ax.set_xticks([-PI, 0, PI])
    ax.set_xticklabels([r"$-\pi$", "0", r"$\pi$"])
    ax.set_yticks([-PI, 0, PI])
    ax.set_yticklabels([r"$-\pi$", "0", r"$\pi$"])


def draw_workspace(ax, title="Workspace"):
    ax.set_title(title)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(mpatches.Circle((0, 0), L1 + L2, fc="none", ec="#bbb", lw=0.6, ls="--"))
    ax.add_patch(mpatches.Circle((0, 0), abs(L1 - L2), fc="none", ec="#bbb", lw=0.6, ls="--"))
    for c, r in obstacles_ws:
        ax.add_patch(mpatches.Circle(c, r, fc=C_OBS, ec=C_OBS_E, lw=1))
    ax.plot(0, 0, "s", color="k", ms=4, zorder=10)


# --- Sample collision-free configurations ---
N = 5000
sampled_points = []
while len(sampled_points) < N:
    t1 = np.random.uniform(-PI, PI)
    t2 = np.random.uniform(-PI, PI)
    if is_free(t1, t2):
        sampled_points.append((t1, t2))

sampled_points = np.array(sampled_points)
print(f"Sampled {N} collision-free configurations")


# --- Dual-panel visualization ---
fig, (ax_ws, ax_cs) = plt.subplots(1, 2, figsize=(8, 3.8),
                                     gridspec_kw={"width_ratios": [1, 1.15]})

draw_workspace(ax_ws)
rng = np.random.default_rng(3)
for idx in rng.choice(N, 15, replace=False):
    t1, t2 = sampled_points[idx]
    elbow, ee = forward_kin(t1, t2)
    ax_ws.plot([0, elbow[0], ee[0]], [0, elbow[1], ee[1]], "-o",
               color="C0", lw=1.2, ms=2, alpha=0.2, solid_capstyle="round")

ax_cs.set_title("Configuration space (C-space)")
draw_cspace_bg(ax_cs)
ax_cs.scatter(sampled_points[:, 0], sampled_points[:, 1],
              c="C3", s=0.4, alpha=0.3, edgecolors="none", rasterized=True)
style_cspace_ax(ax_cs)

plt.tight_layout(); plt.show()

### Visualizing the conditional paths

For a single data point $x_1$ (a valid configuration), the interpolation $x_t = (1-t)\,x_0 + t\,x_1$ traces a straight line from noise to the free C-space. Below we show $t = 0$ (noise), $t = 0.5$, and $t = 1$ (data).

In [ ]:
noise = np.random.randn(N, 2)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, t_val, label in zip(axes, [0.0, 0.5, 1.0],
        [r"$t=0$ (noise $x_0$)", r"$t=0.5$ (interpolated)", r"$t=1$ (data $x_1$)"]):
    pts = (1 - t_val) * noise + t_val * sampled_points
    draw_cspace_bg(ax, alpha=0.45)
    c = "C0" if t_val < 1 else "C3"
    ax.scatter(pts[:, 0], pts[:, 1], s=0.5, alpha=0.3, c=c, edgecolors="none", rasterized=True)
    ax.set_title(label)
    ax.set_xlim(*PLIM); ax.set_ylim(*PLIM)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

ax_mid = axes[1]
rng = np.random.default_rng(0)
for idx in rng.choice(N, 15, replace=False):
    ts = np.linspace(0, 1, 30)
    path = np.outer(1 - ts, noise[idx]) + np.outer(ts, sampled_points[idx])
    ax_mid.plot(path[:, 0], path[:, 1], "-", color="C1", lw=0.4, alpha=0.35)

fig.suptitle(r"Conditional interpolation $x_t = (1-t)\,x_0 + t\,x_1$", y=1.01)
plt.tight_layout(); plt.show()

## 4. Model

An MLP with sinusoidal time embeddings (same idea as positional encodings in transformers). The time $t$ is projected to a high-dimensional embedding and added to the data projection, then passed through residual-free blocks.

$$v_\theta(x_t, t) = \text{MLP}\big(\text{proj}(x_t) + \text{proj}_t(\gamma(t))\big)$$

where $\gamma(t)$ is the sinusoidal embedding.

In [ ]:
class Block(nn.Module):
    def __init__(self, channels=512):
        super().__init__()
        self.ff = nn.Linear(channels, channels)
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.ff(x))


class FlowMLP(nn.Module):
    def __init__(self, dim_data=2, layers=5, channels=512, dim_t=512):
        super().__init__()
        self.dim_t = dim_t
        self.in_proj = nn.Linear(dim_data, channels)
        self.t_proj = nn.Linear(dim_t, channels)
        self.blocks = nn.Sequential(*[Block(channels) for _ in range(layers)])
        self.out_proj = nn.Linear(channels, dim_data)

    def sinusoidal_embedding(self, t, max_positions=10000):
        """Maps scalar t ∈ [0,1] → high-dimensional sinusoidal features."""
        t = t * max_positions
        half = self.dim_t // 2
        freqs = torch.arange(half, device=t.device).float()
        freqs = freqs.mul(-math.log(max_positions) / (half - 1)).exp()
        args = t[:, None] * freqs[None, :]
        emb = torch.cat([args.sin(), args.cos()], dim=1)
        if self.dim_t % 2 == 1:
            emb = nn.functional.pad(emb, (0, 1))
        return emb

    def forward(self, x, t):
        x = self.in_proj(x)
        t = self.sinusoidal_embedding(t)
        t = self.t_proj(t)
        x = x + t  # additive conditioning
        x = self.blocks(x)
        return self.out_proj(x)


model = FlowMLP(layers=5, channels=512).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Model parameters: 1,578,498


## 5. Training

Each iteration:
1. Sample a mini-batch of data points $x_1$
2. Sample noise $x_0 \sim \mathcal{N}(0, I)$ and time $t \sim \mathcal{U}[0, 1]$
3. Build the interpolant $x_t = (1-t)\,x_0 + t\,x_1$
4. The target velocity is $u = x_1 - x_0$
5. Minimize $\| v_\theta(x_t, t) - u \|^2$

In [ ]:
data = torch.tensor(sampled_points, dtype=torch.float32).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-4)

training_steps = 100_000
batch_size = 256
losses = []

pbar = tqdm.tqdm(range(training_steps))
for step in pbar:
    # 1) sample data x1
    x1 = data[torch.randint(len(data), (batch_size,))]
    # 2) sample noise x0 and time t
    x0 = torch.randn_like(x1)
    t = torch.rand(batch_size, device=device)
    # 3) interpolate
    xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    # 4) target velocity
    target = x1 - x0
    # 5) predict & loss
    pred = model(xt, t)
    loss = ((pred - target) ** 2).mean()

    optim.zero_grad()
    loss.backward()
    optim.step()

    losses.append(loss.item())
    if step % 500 == 0:
        pbar.set_postfix(loss=f"{loss.item():.4f}")

100%|██████████| 100000/100000 [05:18<00:00, 313.91it/s, loss=2.8378]


In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.2))
window = 500
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
ax.plot(smoothed, color="C3", lw=0.8)
ax.set_xlabel("Step"); ax.set_ylabel("MSE loss")
ax.set_title("Training loss (smoothed)")
ax.set_xlim(0, len(smoothed))
plt.tight_layout(); plt.show()

## 6. Sampling — Euler ODE integration

Starting from $x_0 \sim \mathcal{N}(0, I)$, we integrate forward:

$$x_{t+\Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t), \qquad \Delta t = \tfrac{1}{K}$$

We store snapshots at each step to animate the transport from noise to data.

In [ ]:
torch.manual_seed(42)
model.eval()

n_samples = 2000
euler_steps = 100
dt = 1.0 / euler_steps

xt = torch.randn(n_samples, 2, device=device)
snapshots = [xt.cpu().numpy().copy()]

with torch.no_grad():
    for i, t_val in enumerate(torch.linspace(0, 1, euler_steps, device=device)):
        vel = model(xt, t_val.expand(n_samples))
        xt = xt + dt * vel
        snapshots.append(xt.cpu().numpy().copy())

model.train()
print(f"Collected {len(snapshots)} snapshots ({euler_steps} Euler steps)")

Collected 101 snapshots (100 Euler steps)


### Static snapshots at key timesteps

In [ ]:
show_steps = [0, 10, 25, 50, 75, 100]
fig, axes = plt.subplots(1, len(show_steps), figsize=(14, 2.5))

for ax, si in zip(axes, show_steps):
    pts = snapshots[si]
    t_label = si / euler_steps
    draw_cspace_bg(ax, alpha=0.45)
    color = "C1" if si == euler_steps else "C0"
    ax.scatter(pts[:, 0], pts[:, 1], s=0.4, alpha=0.4, c=color,
               edgecolors="none", rasterized=True)
    ax.set_title(f"$t={t_label:.2f}$")
    ax.set_xlim(*PLIM); ax.set_ylim(*PLIM)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(r"ODE integration snapshots (noise $\rightarrow$ free C-space)", y=1.02)
plt.tight_layout(); plt.show()

## 7. Animated sampling

Watch the full transport from Gaussian noise to the collision-free C-space. The animation also traces a few individual particle trajectories so you can see the ODE paths.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=80)
plt.rcParams["animation.embed_limit"] = 50
ax.set_xlim(*PLIM); ax.set_ylim(*PLIM)
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])

draw_cspace_bg(ax, alpha=0.5)

scat = ax.scatter([], [], s=1, alpha=0.4, c="C0", edgecolors="none")
title = ax.set_title("", fontsize=8, pad=4)

trace_ids = np.random.default_rng(7).choice(n_samples, 25, replace=False)
trace_lines = []
for _ in trace_ids:
    ln, = ax.plot([], [], "-", color="C1", lw=0.4, alpha=0.35)
    trace_lines.append(ln)

bar_y = -PI - 0.15
ax.plot([-PI, PI], [bar_y, bar_y], "-", color="#ddd", lw=2.5, solid_capstyle="round")
bar_line, = ax.plot([-PI, -PI], [bar_y, bar_y], "-", color="C3", lw=2.5, solid_capstyle="round")

total_frames = len(snapshots) + 20

def update(frame):
    si = min(frame, len(snapshots) - 1)
    t_val = si / euler_steps
    pts = snapshots[si]

    scat.set_offsets(pts)
    scat.set_facecolors("C1" if si >= euler_steps else "C0")

    for ln, idx in zip(trace_lines, trace_ids):
        history = np.array([snapshots[s][idx] for s in range(si + 1)])
        ln.set_data(history[:, 0], history[:, 1])

    progress = min(t_val, 1.0)
    bar_line.set_data([-PI, -PI + progress * 2 * PI], [bar_y, bar_y])

    status = "done" if si >= euler_steps else f"t = {t_val:.2f}"
    title.set_text(f"Flow Matching ODE sampling   [{status}]")
    return scat, title, bar_line, *trace_lines

anim = FuncAnimation(fig, update, frames=total_frames, interval=50, blit=False)
plt.close(fig)

HTML(f'<div style="display:flex;justify-content:center;">{anim.to_jshtml()}</div>')

## 8. Learned velocity field

We can visualize what the network learned by evaluating $v_\theta(x, t)$ on a grid at different timesteps. Early on ($t \approx 0$) the field points broadly toward data modes; near $t = 1$ it makes fine corrections.

In [ ]:
model.eval()

grid_res = 22
gx = np.linspace(-PI, PI, grid_res)
gy = np.linspace(-PI, PI, grid_res)
GX, GY = np.meshgrid(gx, gy)
grid_pts = torch.tensor(np.column_stack([GX.ravel(), GY.ravel()]),
                         dtype=torch.float32, device=device)

t_values = [0.0, 0.25, 0.5, 0.75]
fig, axes = plt.subplots(1, len(t_values), figsize=(14, 3.5))

with torch.no_grad():
    for ax, t_val in zip(axes, t_values):
        t_tensor = torch.full((len(grid_pts),), t_val, device=device)
        vel = model(grid_pts, t_tensor).cpu().numpy()
        U = vel[:, 0].reshape(grid_res, grid_res)
        V = vel[:, 1].reshape(grid_res, grid_res)
        mag = np.sqrt(U**2 + V**2)

        draw_cspace_bg(ax, alpha=0.4)
        ax.quiver(GX, GY, U, V, mag, cmap="RdYlBu_r", alpha=0.75, scale=80, width=0.005)
        ax.set_title(f"$v_\\theta(x,\\, t={t_val:.2f})$")
        ax.set_xlim(*PLIM); ax.set_ylim(*PLIM)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Learned velocity field at different timesteps", y=1.02)
plt.tight_layout(); plt.show()
model.train();

## 9. Generated samples as arm configurations

The real test: do the generated $(\theta_1, \theta_2)$ samples correspond to **valid, collision-free** arm poses?

We take the final ODE samples, check each one against the actual collision geometry, and visualize the resulting arm configurations in the workspace. Green arms are collision-free; red ones collide with an obstacle.

In [ ]:
# --- Evaluate collision-free rate ---
generated = snapshots[-1]
valid_mask = np.array([is_free(t1, t2) for t1, t2 in generated])
n_valid = valid_mask.sum()
n_total = len(generated)
rate = n_valid / n_total * 100
print(f"Collision-free rate: {n_valid}/{n_total}  ({rate:.1f}%)")


# --- 3-panel figure ---
fig, (ax_cs, ax_good, ax_bad) = plt.subplots(
    1, 3, figsize=(12, 4), gridspec_kw={"width_ratios": [1.1, 1, 1]}
)

# Panel 1: C-space colored by validity
draw_cspace_bg(ax_cs, alpha=0.55)
ax_cs.scatter(generated[~valid_mask, 0], generated[~valid_mask, 1],
              c="C3", s=2, alpha=0.6, edgecolors="none", zorder=2)
ax_cs.scatter(generated[valid_mask, 0], generated[valid_mask, 1],
              c="C1", s=3, alpha=0.5, edgecolors="none", zorder=3)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='C1',
           markersize=8, markeredgecolor='black', markeredgewidth=0.5, label='free'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='C3',
           markersize=8, markeredgecolor='black', markeredgewidth=0.5, label='collision'),
]
ax_cs.legend(handles=legend_handles, loc="upper right", framealpha=1.0,
             edgecolor="gray", fancybox=False)

style_cspace_ax(ax_cs)
ax_cs.set_title(f"Generated samples --- {rate:.1f}\\% collision-free")

# Panel 2: Workspace — valid arms
draw_workspace(ax_good, title=f"Valid configurations ({n_valid})")
rng = np.random.default_rng(12)
valid_idx = np.where(valid_mask)[0]
for idx in rng.choice(valid_idx, min(40, len(valid_idx)), replace=False):
    t1, t2 = generated[idx]
    elbow, ee = forward_kin(t1, t2)
    ax_good.plot([0, elbow[0], ee[0]], [0, elbow[1], ee[1]], "-o",
                 color="C1", lw=1, ms=1.5, alpha=0.15, solid_capstyle="round")
for idx in rng.choice(valid_idx, min(5, len(valid_idx)), replace=False):
    t1, t2 = generated[idx]
    elbow, ee = forward_kin(t1, t2)
    ax_good.plot([0, elbow[0], ee[0]], [0, elbow[1], ee[1]], "-o",
                 color="C1", lw=2, ms=3, alpha=0.7, solid_capstyle="round", zorder=5)
    ax_good.plot(*ee, "o", color="C1", ms=4, mec="white", mew=0.6, zorder=6)

# Panel 3: Workspace — colliding arms
invalid_idx = np.where(~valid_mask)[0]
n_inv = len(invalid_idx)
draw_workspace(ax_bad, title=f"Colliding configurations ({n_inv})")
if n_inv > 0:
    for idx in rng.choice(invalid_idx, min(30, n_inv), replace=False):
        t1, t2 = generated[idx]
        elbow, ee = forward_kin(t1, t2)
        ax_bad.plot([0, elbow[0], ee[0]], [0, elbow[1], ee[1]], "-o",
                    color="C3", lw=1, ms=1.5, alpha=0.2, solid_capstyle="round")
    for idx in rng.choice(invalid_idx, min(3, n_inv), replace=False):
        t1, t2 = generated[idx]
        elbow, ee = forward_kin(t1, t2)
        ax_bad.plot([0, elbow[0], ee[0]], [0, elbow[1], ee[1]], "-o",
                    color="C3", lw=2, ms=3, alpha=0.7, solid_capstyle="round", zorder=5)
        ax_bad.plot(*ee, "x", color="C3", ms=6, mew=1.5, zorder=6)
else:
    ax_bad.text(0, 0, "None!", ha="center", va="center", fontsize=12, color="C1")

plt.tight_layout()
plt.show()

### Baseline comparison: random sampling vs flow matching

If we sampled $(\theta_1, \theta_2)$ uniformly at random, the collision rate would equal the fraction of C-space occupied by obstacles. The flow model learns to avoid those regions entirely.

In [ ]:
# Ground truth from the precomputed grid
obstacle_frac = (~free_grid).sum() / free_grid.size
random_collision_rate = obstacle_frac * 100
flow_collision_rate = 100 - rate

print(f"C-space obstacle coverage:      {obstacle_frac:.1%}")
print(f"Random uniform collision rate:  {random_collision_rate:.1f}%")
print(f"Flow matching collision rate:    {flow_collision_rate:.1f}%")
print(f"Improvement factor:              {random_collision_rate / max(flow_collision_rate, 0.1):.1f}x")

fig, ax = plt.subplots(figsize=(4, 2.8))
labels = ["Random\nuniform", "Flow\nmatching"]
vals = [100 - random_collision_rate, rate]

bars = ax.bar(labels, vals, width=0.45, color=["C0", "C1"],
              edgecolor="black", linewidth=0.4)
ax.set_ylabel("Collision-free rate (\\%)")
ax.set_ylim(0, 115)

for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 3,
            f"{v:.1f}\\%", ha="center", va="bottom")

plt.tight_layout()
plt.show()